# 📚 Bookify — Engineering Book Recommendation System
## End-to-End ML Notebook

This notebook walks through the complete recommendation system pipeline.
Every concept is explained in plain English — no ML background required.

**Before running this notebook**, generate the clean dataset:
```bash
python scripts/preprocess.py
```

Sections:
1. Setup
2. Load & Explore Data
3. Preprocessing & Category Assignment
4. TF-IDF Feature Engineering
5. Cosine Similarity
6. Bayesian Hybrid Scoring
7. Evaluation & Visualisations
8. Live Recommendation Demo
9. Using a Saved Model


## 1. Setup

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')
plt.rcParams['figure.facecolor'] = '#181818'
plt.rcParams['axes.facecolor']   = '#1e1e1e'
plt.rcParams['text.color']       = 'white'
plt.rcParams['axes.labelcolor']  = '#b3b3b3'
plt.rcParams['xtick.color']      = '#b3b3b3'
plt.rcParams['ytick.color']      = '#b3b3b3'
print("✅ Libraries imported")


## 2. Load & Explore Data

Load the clean merged dataset produced by `scripts/preprocess.py`.

In [ ]:
# Load the clean merged dataset (run scripts/preprocess.py first)
ROOT = os.path.dirname(os.path.abspath(''))  # adjust if needed
CLEAN_CSV = 'outputs/books_clean.csv'

df = pd.read_csv(CLEAN_CSV)
print(f"Books loaded: {len(df):,}")
print(f"Columns: {list(df.columns)}")
df[['title','author','rating','ratings_count','year','download_link','price']].head(5)


In [ ]:
# Rating distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
rated = df[df['rating'] > 0]

axes[0].hist(rated['rating'], bins=20, color='#1db954', edgecolor='#121212')
axes[0].set_title('Rating Distribution', color='white')
axes[0].set_xlabel('Rating')

axes[1].hist(np.log1p(df['ratings_count']), bins=25, color='#a78bfa', edgecolor='#121212')
axes[1].set_title('Log(Ratings Count) Distribution', color='white')
axes[1].set_xlabel('log(1 + ratings_count)')

plt.tight_layout(); plt.show()
print(f"Books with ratings: {(df['rating']>0).sum():,} / {len(df):,}")
print(f"Books with download links: {(df['download_link']!='').sum():,}")
print(f"Books with price data: {(df['price']>0).sum():,}")


In [ ]:
# Missing values per column
missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]

fig, ax = plt.subplots(figsize=(9, 3))
ax.barh(missing.index, missing.values, color='#f87171', edgecolor='#121212')
ax.set_title('Missing Values per Column', color='white')
ax.set_xlabel('Count')
plt.tight_layout(); plt.show()
print(missing)


## 3. Category Distribution

In [ ]:
print(df['category'].value_counts().to_string())
counts = df['category'].value_counts()

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#1db954','#a78bfa','#60a5fa','#f87171','#34d399',
          '#fb923c','#f472b6','#fbbf24','#94a3b8','#c084fc','#67e8f9','#6a6a6a','#f59e0b']
ax.barh(counts.index, counts.values, color=colors[:len(counts)], edgecolor='#121212')
ax.set_title('Books by Category', color='white')
ax.set_xlabel('Count')
plt.tight_layout(); plt.show()


## 4. TF-IDF Feature Engineering

**What is TF-IDF?**

**TF** = Term Frequency — how often a word appears in *this* book's text.  
**IDF** = Inverse Document Frequency — how rare that word is across *all* books.  
**TF-IDF** = TF × IDF — high score means the word is frequent here but rare elsewhere.

Example: "algorithm" appearing 20 times in a data-structures book gets a high TF-IDF  
because it's distinctive for that book — not every book uses it as heavily.

We combine `title + author + description` into one text blob per book,  
then TF-IDF converts it into a vector of 8,000 numbers.


In [ ]:
df['content'] = df['title'] + ' ' + df['author'] + ' ' + df['description'].fillna('')

vectorizer = TfidfVectorizer(
    max_features = 8000,      # keep the 8,000 most informative terms
    stop_words   = 'english', # ignore 'the', 'and', 'is', etc.
    ngram_range  = (1, 2),    # unigrams AND bigrams (e.g. 'machine learning')
    sublinear_tf = True,      # log-scale TF to dampen very frequent words
    min_df       = 2,         # ignore terms in fewer than 2 books
)
tfidf_matrix = vectorizer.fit_transform(df['content'])
print(f"TF-IDF matrix: {tfidf_matrix.shape[0]:,} books × {tfidf_matrix.shape[1]:,} features")
print(f"Matrix is {100 * tfidf_matrix.nnz / (tfidf_matrix.shape[0]*tfidf_matrix.shape[1]):.2f}% non-zero (very sparse)")


In [ ]:
# Top TF-IDF terms for a sample book
idx   = 0
title = df.iloc[idx]['title']
names = vectorizer.get_feature_names_out()
scores = tfidf_matrix[idx].toarray()[0]
top_idx = scores.argsort()[::-1][:12]

print(f"Top TF-IDF terms for: '{title}'\n")
for i in top_idx:
    if scores[i] > 0:
        print(f"  {names[i]:<35} {scores[i]:.4f}")


## 5. Cosine Similarity

Each book is now a point in 8,000-dimensional space.  
**Cosine similarity** measures the angle between two book-vectors:

- **1.0** → same direction → very similar content  
- **0.0** → perpendicular → completely different content  

We pre-compute the full N×N matrix so any book→similar-books lookup is O(1) at query time.


In [ ]:
print("Computing cosine similarity matrix (may take 10-30s)...")
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f"Similarity matrix: {cosine_sim.shape}  ({cosine_sim.nbytes/1e6:.0f} MB)")
print(f"Diagonal (self-similarity): {cosine_sim.diagonal().mean():.4f}  (should be 1.0)")


In [ ]:
# Example: most similar books to book[0]
query_idx   = 0
query_title = df.iloc[query_idx]['title']

sim_scores = sorted(enumerate(cosine_sim[query_idx]), key=lambda x: x[1], reverse=True)[1:6]
print(f"Books most similar to: '{query_title}'\n")
for rank, (idx, score) in enumerate(sim_scores, 1):
    print(f"  {rank}. {df.iloc[idx]['title'][:65]}")
    print(f"       sim={score:.4f}  rating={df.iloc[idx]['rating']}")


## 6. Bayesian Hybrid Scoring

**Why Bayesian average instead of raw rating?**

A book with 3 ratings of 5.0 should NOT outrank a book with 5,000 ratings at 4.2.  
The Bayesian formula pulls low-vote books toward the global mean — similar to IMDb's  
weighted rating formula.

```
WR = (v / (v+m)) × R + (m / (v+m)) × C
```
- `v` = book's ratings count  
- `m` = minimum vote threshold (500)  
- `R` = book's average rating  
- `C` = global mean rating  

Then we combine with log-normalised popularity:  
```
score = 0.60 × bayes_norm + 0.40 × log(count)_norm
```


In [ ]:
df_scored = df.copy()
rated = df_scored[df_scored['rating'] > 0]
C = rated['rating'].mean()   # global mean
m = 500                      # minimum votes threshold

v = df_scored['ratings_count']
df_scored['bayes_rating'] = (v / (v+m)) * df_scored['rating'] + (m / (v+m)) * C
df_scored.loc[df_scored['rating'] == 0, 'bayes_rating'] = 0

df_scored['log_count'] = np.log1p(df_scored['ratings_count'])
scaler = MinMaxScaler()
scaled = scaler.fit_transform(df_scored[['bayes_rating', 'log_count']])
df_scored['score'] = 0.60 * scaled[:, 0] + 0.40 * scaled[:, 1]
df_scored.loc[df_scored['rating'] == 0, 'score'] = 0

# Re-scale within top-N for meaningful ranking
top_books = df_scored.nlargest(500, 'score').reset_index(drop=True)
s_min, s_max = top_books['score'].min(), top_books['score'].max()
top_books['score'] = ((top_books['score'] - s_min) / (s_max - s_min)).round(4)

print(f"Top 500 books — avg rating: {top_books[top_books['rating']>0]['rating'].mean():.2f}")
print(f"Score range: {top_books['score'].min():.4f} – {top_books['score'].max():.4f}  std: {top_books['score'].std():.4f}")
top_books[['title','author','rating','ratings_count','score','category']].head(10)


## 7. Evaluation & Visualisations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Category pie
counts = top_books['category'].value_counts()
colors = ['#1db954','#a78bfa','#60a5fa','#f87171','#34d399',
          '#fb923c','#f472b6','#fbbf24','#94a3b8','#c084fc','#67e8f9','#6a6a6a','#f59e0b']
axes[0].pie(counts.values, labels=counts.index, colors=colors[:len(counts)],
            autopct='%1.0f%%', textprops={'color':'white','fontsize':8})
axes[0].set_title('Top 500 by Category', color='white')

# Score vs rating scatter
rated = top_books[top_books['rating'] > 0]
sc = axes[1].scatter(rated['rating'], rated['score'],
                     c=np.log1p(rated['ratings_count']), cmap='YlGn',
                     alpha=0.8, edgecolors='#121212', s=50)
plt.colorbar(sc, ax=axes[1], label='log(ratings count)')
axes[1].set_xlabel('Rating'); axes[1].set_ylabel('ML Score')
axes[1].set_title('ML Score vs Goodreads Rating', color='white')

plt.tight_layout(); plt.show()


In [ ]:
# Top 10 by ML score
top10 = top_books.head(10)[['title','score','rating','category']].copy()
top10['short'] = top10['title'].str[:45] + '…'

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(top10['short'][::-1], top10['score'][::-1], color='#1db954', edgecolor='#121212')
for bar, val in zip(bars, top10['score'][::-1]):
    ax.text(val+0.005, bar.get_y()+bar.get_height()/2,
            f'{val:.3f}', va='center', color='#b3b3b3', fontsize=9)
ax.set_title('Top 10 Books by ML Score', color='white')
ax.set_xlabel('Score (0–1, re-scaled within top-500)')
plt.tight_layout(); plt.show()


## 8. Live Recommendation Demo

In [ ]:
def recommend(query_title: str, top_n: int = 5) -> pd.DataFrame:
    """Given a partial title, return the N most similar books."""
    matches = df[df['title'].str.lower().str.contains(query_title.lower())]
    if matches.empty:
        print(f"Not found: '{query_title}'")
        return pd.DataFrame()

    idx   = matches.index[0]
    found = df.iloc[idx]['title']
    print(f"Query : {found}")
    print(f"Author: {df.iloc[idx]['author']}")
    print(f"Rating: {df.iloc[idx]['rating']}  |  Download: {bool(df.iloc[idx].get('download_link',''))}\n")

    sims = sorted(enumerate(cosine_sim[idx]), key=lambda x: x[1], reverse=True)[1:top_n+1]
    return pd.DataFrame([{
        'Rank': i+1,
        'Title': df.iloc[j]['title'][:55],
        'Author': df.iloc[j]['author'][:22],
        'Rating': df.iloc[j]['rating'],
        'Similarity': round(s, 3),
        'Category': df.iloc[j]['category'],
    } for i, (j, s) in enumerate(sims)])


In [ ]:
recommend('Software Engineering at Google')

In [ ]:
recommend('machine learning')

In [ ]:
recommend('Embedded Systems')

## 9. Using a Saved Model on New Text

In [ ]:
loaded_vec = joblib.load('models/tfidf_vectorizer.pkl')

# Get recommendations for arbitrary text — no retraining needed
query = "database indexing query optimisation sql performance"
q_vec = loaded_vec.transform([query])
q_sim = cosine_similarity(q_vec, tfidf_matrix)[0]

top_idx = q_sim.argsort()[::-1][:5]
print(f"Books matching: '{query}'\n")
for i in top_idx:
    print(f"  • {df.iloc[i]['title'][:65]}")
    print(f"    sim={q_sim[i]:.4f}  rating={df.iloc[i]['rating']}  cat={df.iloc[i]['category']}")


## Summary

| Component | Detail |
|---|---|
| Source data | Goodreads (2,375) + Google Books (1,488) |
| After fuzzy dedup + domain filter | ~3,233 unique engineering books |
| Model | Content-Based Filtering |
| Features | TF-IDF on title + author + description |
| Vocabulary | 8,000 tokens (unigrams + bigrams) |
| Similarity | Pairwise cosine similarity (N×N matrix) |
| Scoring | Bayesian average + log-popularity, re-scaled within top-N |
| Top books | 500 (raised from 150 to ensure similar-book links work) |
| Recs per book | 5 (restricted to top-500 pool) |
| Download links | ~2,200+ books have direct PDF/EPUB links |
